# Evaluation of Fusion Algorithms

Miguel Angel Ramírez

22.01.2025

This notebook aims to evaluate orthoimages for the function experiments between PlanetScope and Sentinel-2.

This notebook:
- Read the bands and calculate the quality indicators.
- Stack the images for the next step of classification.

In [2]:
import numpy as np
import os
import rasterio
import os
import matplotlib.pyplot as plt
import pandas as pd
import csv

from osgeo import gdal, gdalconst
from rasterio import Affine
from rasterio.enums import Resampling
from rasterio.merge import merge

In [3]:
def load_image(image_path):
    """
    Carga una imagen desde la ruta de archivo especificada.

    Args:
    image_path (str): Ruta de archivo de la imagen.

    Returns:
    numpy.ndarray: La imagen cargada como una matriz NumPy.
    """
    with rasterio.open(image_path) as ds:
        return ds.read(1)

def calculate_rmse(original_image, recovered_image):
    # Calcula el Root Mean Square Error (RMSE) entre dos imágenes
    if original_image.shape != recovered_image.shape:
        return np.nan
    return np.sqrt(np.mean((original_image - recovered_image) ** 2))

def calculate_ergas(original_image, recovered_image, scaling_factor=2):
    # Calcula el Error Relativo Global en la Escala de Gris (ERGAS) entre dos imágenes
    rmse = calculate_rmse(original_image, recovered_image)
    k = scaling_factor  # Factor de escalado
    return (rmse / k) / np.sqrt(np.mean(original_image ** 2))

def calculate_cc(original_image, recovered_image):
    # Calcula el Coefficient de Correlation (CC) entre dos imágenes
    if original_image.shape != recovered_image.shape:
        return np.nan
    original_mean = np.mean(original_image)
    recovered_mean = np.mean(recovered_image)
    covar = np.mean((original_image - original_mean) * (recovered_image - recovered_mean))
    original_std = np.std(original_image)
    recovered_std = np.std(recovered_image)
    return covar / (original_std * recovered_std)

def calculate_q(original_image, recovered_image, c=2):
    # Calcula el Índice de Calidad Universal (Q) entre dos imágenes
    cc = calculate_cc(original_image, recovered_image)
    ergas = calculate_ergas(original_image, recovered_image)
    return c * cc / (1 + ergas)

def calculate_metrics_by_band(original_folder, methods_folder, band_numbers, scaling_factor=3, c=3, output_file=None):
    band_metrics = []  # List to store metrics per band and method

    for band_num in band_numbers:
        original_path = os.path.join(original_folder, f"band{band_num}.tif")
        original_image = load_image(original_path)

        for method in os.listdir(methods_folder):
            # Check if the current method file corresponds to the band number
            if method.startswith(f"band{band_num}_"):
                # Extract the method name without the band number
                method_name = os.path.splitext(method)[0].replace(f"band{band_num}_", "")
                recovered_image = load_image(os.path.join(methods_folder, method))

                rmse = calculate_rmse(original_image, recovered_image)
                ergas = calculate_ergas(original_image, recovered_image, scaling_factor)
                cc = calculate_cc(original_image, recovered_image)
                q = calculate_q(original_image, recovered_image, c)

                metrics = {
                    "band": band_num,
                    "method": method_name,
                    "rmse": rmse,
                    "ergas": ergas,
                    "cc": cc,
                    "q": q
                }

                band_metrics.append(metrics)

    if output_file:
        with open(output_file, 'w', newline='') as csvfile:
            fieldnames = ["band", "method", "rmse", "ergas", "cc", "q"]
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(band_metrics)

    return band_metrics


def stack_fusion_images(fusion_folder, output_folder, method):
    # Crear la carpeta de salida si no existe
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Crear una lista con las imágenes fusionadas correspondientes al método
    fusion_images = [f for f in os.listdir(fusion_folder) if f.endswith(f"{method}_fusion.tif")]

    if not fusion_images:
        print(f"No se encontraron imágenes de fusión para el método {method}.")
        return

    # Ordenar las imágenes por nombre de banda
    fusion_images.sort(key=lambda x: int(x.split("_")[0].replace("band", "")))

    # Abrir la primera imagen para obtener información de georreferenciación
    first_image = os.path.join(fusion_folder, fusion_images[0])
    ds = gdal.Open(first_image, gdal.GA_ReadOnly)

    # Obtener información de georreferenciación
    width = ds.RasterXSize
    height = ds.RasterYSize
    num_bands = len(fusion_images)

    # Crear un archivo de salida para el stack layer
    output_filename = os.path.join(output_folder, f"S2_{method}_fusion_stack.tif")
    driver = gdal.GetDriverByName("GTiff")
    stack_ds = driver.Create(output_filename, width, height, num_bands, gdal.GDT_Float32)

    if stack_ds is None:
        print("Error al crear el archivo de salida.")
        return

    # Configurar opciones de compresión
    compression_options = ['COMPRESS=LZW', 'PREDICTOR=2']

    # Copiar la información de georreferenciación y proyección
    stack_ds.SetGeoTransform(ds.GetGeoTransform())
    stack_ds.SetProjection(ds.GetProjection())

    # Aplicar compresión a las bandas
    for band_num in range(1, num_bands + 1):
        stack_band = stack_ds.GetRasterBand(band_num)
        
        # Configurar opciones de compresión
        for option in compression_options:
            stack_band.SetMetadata({option: option}, 'IMAGE_STRUCTURE')

        fusion_image = os.path.join(fusion_folder, fusion_images[band_num - 1])
        fusion_ds = gdal.Open(fusion_image, gdal.GA_ReadOnly)
        fusion_band = fusion_ds.GetRasterBand(1)
        stack_band.WriteArray(fusion_band.ReadAsArray())

        fusion_ds = None

    stack_ds = None
    print(f"Imagen de stack layer para el método {method} creada en {output_filename}.")

def stack_selected_bands(bands_folder, output_path, compression_options=None):
    # Obtener una lista de las rutas de los archivos de bandas en la carpeta
    band_files = [os.path.join(bands_folder, f) for f in os.listdir(bands_folder) if f.endswith(".tif")]

    # Filtrar las bandas específicas que deseas apilar
    selected_band_numbers = [2, 3, 4, 5, 6, 7, 8, 10, 11, 12]

    # Crear una lista para almacenar las bandas seleccionadas
    selected_bands = []

    projection = None  # Inicializar la proyección como nula
    transform = None  # Inicializar la transformación como nula

    for band_num in selected_band_numbers:
        matching_band_file = next((f for f in band_files if f.endswith(f"band{band_num}.tif")), None)

        if matching_band_file is not None:
            with rasterio.open(matching_band_file, 'r') as ds:
                selected_band = ds.read(1)
                selected_bands.append(selected_band)

                if projection is None:
                    projection = ds.crs
                elif projection != ds.crs:
                    print("Advertencia: Las bandas tienen proyecciones diferentes.")

                if transform is None:
                    transform = ds.transform

    if not selected_bands:
        print("No se encontraron bandas seleccionadas para apilar.")
        return

    # Crear la imagen de stack layer con las bandas seleccionadas
    num_bands = len(selected_bands)
    height, width = selected_bands[0].shape

    # Crear el archivo de salida con las opciones de compresión
    with rasterio.open(output_path, 'w', driver='GTiff', height=height, width=width, count=num_bands,
                       dtype=rasterio.float32, crs=projection, transform=transform, compress=compression_options) as stack_ds:
        for i in range(num_bands):
            stack_ds.write(selected_bands[i], i + 1)

    print(f"Imagen de stack layer creada en {output_path} con proyección asignada.")

def stack_rasters(input_paths, output_path):
    # Abrir los archivos raster de entrada
    src_files_to_mosaic = []
    for path in input_paths:
        src = rasterio.open(path)
        src_files_to_mosaic.append(src)

    # Combinar los rasters en una sola imagen apilada
    mosaic, out_trans = merge(src_files_to_mosaic)

    # Guardar el raster apilado en el archivo de salida
    with rasterio.open(output_path, "w", driver="GTiff", width=mosaic.shape[2], height=mosaic.shape[1], count=mosaic.shape[0], dtype=mosaic.dtype, crs=src.crs, transform=out_trans) as dest:
        for i in range(mosaic.shape[0]):
            dest.write(mosaic[i], i + 1)

    print(f'Rasters apilados y guardados en {output_path}')

# Quality Fusion Indicators

In [ ]:
original_folder = "TOLIMA 2023/S2_bands"
methods_folder = "TOLIMA 2023/S2_f"
band_numbers = [2, 3, 4, 5, 6, 7, 8, 10, 11, 12]
scaling_factor = 1
c = 1

metrics = calculate_metrics_by_band(original_folder, methods_folder, band_numbers, scaling_factor, c,output_file= "TOLIMA 2023/MONTES 2023.csv")

# Graphics

In [ ]:
# Assuming you have the 'band_metrics' dictionary result

# Convert the dictionary to a pandas DataFrame for easier plotting
df = pd.DataFrame(metrics)

# Create subplots for each band
for band_num in df['band'].unique():
    df_band = df[df['band'] == band_num]

    # Create a figure for the current band
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Plot performance metrics for each method
    for method in df_band['method'].unique():
        df_method = df_band[df_band['method'] == method]
        ax.plot(df_method['method'], df_method['rmse'], label=method)
    
    # Customize the plot
    ax.set_title(f'Band {band_num} Performance')
    ax.set_xlabel('Methods')
    ax.set_ylabel('RMSE')
    ax.legend(loc='best')
    
    # Show or save the plot as needed
    plt.show()  # Show the plot
    # To save the plot as an image, use the following line:
    # plt.savefig(f'band{band_num}_performance.png')


# INDEX ANALYSIS

In [ ]:
# Convert the dictionary to a pandas DataFrame
df = pd.DataFrame(metrics)

# Pivot the DataFrame to create the table
pivot_table = df.pivot(index='band', columns='method')

# Display the pivot table
print(pivot_table)

In [ ]:
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show

# Load the original "band11" raster
original_band11_path = "path_to_original_band11.tif"
with rasterio.open(original_band11_path) as src:
    original_band11 = src.read(1)  # Assuming it's a single-band raster

# Create a figure
fig, ax = plt.subplots(figsize=(10, 6))

# Plot the original "band11"
show(original_band11, ax=ax, cmap='viridis', title='Original band11')

# Load and plot the processed results for "band11"
methods_folder = "path_to_processed_results_folder"
method_names = ["method1", "method2", "method3"]  # Update with your method names
for method_name in method_names:
    method_band11_path = f"{methods_folder}/{method_name}_band11_fusion.tif"
    with rasterio.open(method_band11_path) as src:
        method_band11 = src.read(1)  # Assuming it's a single-band raster

    show(method_band11, ax=ax, cmap='viridis', title=f'{method_name} Fusion band11')

# Show the plot
plt.show()


# Get Output Images

In [4]:
# Ejemplo de uso:
fusion_folder = "E:\PLANETSCOPE\MONTES 2011\S2_f"
output_folder = "E:\PLANETSCOPE\MONTES 2011\Final_Image"
#methods = ["rcs", "lmvm", "bayes","avg"]
methods = [ "bayes"]

for method in methods:
    stack_fusion_images(fusion_folder, output_folder, method)

Imagen de stack layer para el método bayes creada en E:\PLANETSCOPE\MONTES 2011\Final_Image\S2_bayes_fusion_stack.tif.
